# Table And Figure Notebook

This notebook keeps analysis logic inside `src/et_severity` and uses the notebook only as a lightweight report runner.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/12Gongsam/Multimodal-ET-Severity-Assessment.git"
REPO_NAME = "Multimodal-ET-Severity-Assessment"

def ensure_repo_checkout():
    cwd = Path.cwd().resolve()
    if cwd.name == "notebook" and (cwd.parent / "README.md").exists():
        repo_root = cwd.parent
    elif (cwd / "README.md").exists() and (cwd / "notebook").exists():
        repo_root = cwd
    elif (cwd / REPO_NAME / "README.md").exists():
        repo_root = cwd / REPO_NAME
    else:
        repo_root = cwd / REPO_NAME
        if not repo_root.exists():
            subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
        elif not (repo_root / "README.md").exists():
            raise FileNotFoundError(f"Repository directory exists but README.md is missing: {repo_root}")

    notebook_dir = repo_root / "notebook"
    if not notebook_dir.exists():
        raise FileNotFoundError(f"Notebook directory not found: {notebook_dir}")

    os.chdir(notebook_dir)
    return repo_root, notebook_dir

REPO_ROOT, NOTEBOOK_DIR = ensure_repo_checkout()
print(f"Repository root: {REPO_ROOT}")
print(f"Working directory: {NOTEBOOK_DIR}")


In [ ]:
%pip install -q -r ../requirements.txt

import sys
from pathlib import Path

SRC_DIR = (Path("..") / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Python executable: {sys.executable}")
print(f"Src path added: {SRC_DIR}")


In [ ]:
import numpy as np
import pandas as pd

from et_severity.analysis import (
    build_session_severity_table_from_csv,
    build_single_vs_multimodal_accuracy_report,
    compare_ordinal_ratings,
    summarize_multimodal_prediction_runs,
    summarize_single_modality_prediction_runs,
    save_table_csv,
)
from et_severity.visualization import (
    plot_gaussian_mixture_fit,
    plot_general_vs_calibration_distribution,
    plot_log_power_boxplot,
    plot_mean_severity_by_tetras,
    plot_patient_power_scatter,
    plot_patient_scatter_with_calibration_band,
    plot_patient_task_scatter_grid,
    plot_single_vs_multimodal_accuracy,
    predict_calibration_labels_from_gmm,
)


# Config


In [ ]:
DATA_DIR = (NOTEBOOK_DIR / ".." / "data").resolve()
PREDICTIONS_DIR = (NOTEBOOK_DIR / ".." / "predictions").resolve()
FIGURE_DIR = (NOTEBOOK_DIR / ".." / "figure").resolve()
LABEL_CSV = DATA_DIR / "relabel_md_k5.csv"
CALIBRATION_CSV = DATA_DIR / "calibration_power_summary.csv"
TABLE4_PATH = FIGURE_DIR / "Table4.csv"

clinical_df = pd.read_csv(LABEL_CSV)
calibration_df = pd.read_csv(CALIBRATION_CSV)
if "log_power" not in calibration_df.columns and "normalized_power" in calibration_df.columns:
    calibration_df["log_power"] = np.log10(calibration_df["normalized_power"])

print(f"Data dir: {DATA_DIR}")
print(f"Predictions dir: {PREDICTIONS_DIR}")
print(f"Figure dir: {FIGURE_DIR}")


# Table 4


In [ ]:
table4 = build_session_severity_table_from_csv(LABEL_CSV)
save_table_csv(table4, TABLE4_PATH)
table4


# Prediction Summaries


In [ ]:
multimodal_summary = summarize_multimodal_prediction_runs(PREDICTIONS_DIR)
multimodal_summary.head()


In [ ]:
single_severity_summary = summarize_single_modality_prediction_runs(
    PREDICTIONS_DIR,
    modality="acc",
    target="severity",
)
single_task_summary = summarize_single_modality_prediction_runs(
    PREDICTIONS_DIR,
    modality="traj",
    target="task",
)

single_severity_summary, single_task_summary


# Figure 5 Style Comparison


In [ ]:
accuracy_report = build_single_vs_multimodal_accuracy_report(PREDICTIONS_DIR)
plot_single_vs_multimodal_accuracy(accuracy_report)
accuracy_report


# Scatter Figures


In [ ]:
plot_patient_power_scatter(clinical_df)


In [ ]:
plot_patient_task_scatter_grid(clinical_df, save_path=FIGURE_DIR / "figure_7_ver3.png")


In [ ]:
plot_patient_scatter_with_calibration_band(clinical_df, calibration_df)


# Supplementary Figures


In [ ]:
plot_log_power_boxplot(
    clinical_df,
    category_col="tetras_score",
    x_label="TETRAS score",
    save_path=FIGURE_DIR / "S_figure1.png",
)


In [ ]:
stat_df, _ = plot_general_vs_calibration_distribution(clinical_df, calibration_df)
stat_df


In [ ]:
gmm, _ = plot_gaussian_mixture_fit(clinical_df, "log_power", k=5)
predicted_calibration = predict_calibration_labels_from_gmm(calibration_df, gmm)
ordinal_stats = compare_ordinal_ratings(predicted_calibration, "tetras_score", "gmm_pred")
ordinal_stats


In [ ]:
plot_log_power_boxplot(
    clinical_df,
    category_col="target_k5",
    label_transform=lambda value: int(value) + 1,
    x_label="Relabeled by GMM",
    save_path=FIGURE_DIR / "S_figure2.png",
)


In [ ]:
plot_mean_severity_by_tetras(table4, save_path=FIGURE_DIR / "S_figure4.png")
